# Pre-Training Configuration

## Purpose
This notebook is run **once by the trainer** before the training starts.
It creates isolated environments for each participant:

1. **Catalog** per user: `retailhub_{username}`
2. **Schemas**: `bronze`, `silver`, `gold`, `default`
3. **Volume**: `datasets` (managed) in the `default` schema
4. **Dataset files**: copied from the Git folder to each user's Volume
5. **Permissions**: full access granted to each participant on their catalog, plus `MANAGE` so they can run the GRANT/REVOKE demo in M06
6. **Demo group check**: verifies that `retailhub_analysts` exists as an *account* group usable in Unity Catalog
7. **Optional**: copies the trainer solutions of `lab_task_02` / `lab_task_03` to a read-only workspace folder (Step 3b)

**After running this notebook**, every module validates the environment with `%run ../setup/00_setup`.

---

### Checklist before running:
- [ ] Databricks workspace is ready (Premium tier, Unity Catalog enabled)
- [ ] Training group exists as an account group with all participants added; its name is set in `TRAINING_GROUP`
- [ ] `STORAGE_LOCATION` is set, or left `""` to use the metastore default storage
- [ ] Trainer logins are listed in `TRAINER_ACCOUNTS`, with the same list in Step 1 here and in `00_setup` in your own Git folder (participants' copies stay empty), or both lists are left empty
- [ ] This repo is cloned as a Git folder in the workspace
- [ ] You have `CREATE CATALOG` and `MANAGE` privileges (account admin or metastore admin)
- [ ] Account group `retailhub_analysts` exists -- create it in **Settings → Identity and access → Groups → Add Group → Add new**. Do **not** use SQL `CREATE GROUP`: it creates a workspace-local group that Unity Catalog cannot grant to

> **Single user / Databricks Free Edition:** set `SINGLE_USER_MODE = True` in Step 0 and run all cells.
> The notebook then creates a catalog only for you, and M06 grants to the built-in `account users` group.
> Without that flag a missing or empty `TRAINING_GROUP` stops the notebook -- a typo never silently switches modes.

## Step 0: Configuration

Set the values marked **SET BEFORE RUNNING**. Do not commit real group names, storage paths or e-mails to the public repo.

In [0]:
# =============================================================================
# CONFIGURATION -- SET BEFORE RUNNING
# =============================================================================

# Account group that contains all training participants (classroom mode)
TRAINING_GROUP = "<training-group>"  # SET BEFORE RUNNING

# Self-study / Databricks Free Edition: no training group -> set to True.
# Creates a catalog only for you; M06 then grants to `account users`.
SINGLE_USER_MODE = False

# Managed storage for the catalogs. "" = metastore default storage.
# Otherwise a path inside an external location on which you have CREATE MANAGED STORAGE, e.g.
#   Azure: "abfss://<container>@<storage-account>.dfs.core.windows.net/<path>"
#   AWS:   "s3://<bucket>/<path>"
STORAGE_LOCATION = ""  # SET BEFORE RUNNING (or leave "")

# Optional (Step 3b): workspace folder with the trainer solutions of lab_task_02/03.
# "" = skip. Never put the solutions into the Git repo.
SOLUTIONS_SOURCE = ""  # e.g. "/Workspace/Users/<trainer-login>/retailhub_solutions_src"
SOLUTIONS_TARGET_NAME = "retailhub_solutions"  # created in your home folder, read-only for TRAINING_GROUP

# Catalog naming: retailhub_{username}
CATALOG_PREFIX = "retailhub"

# Schema names (Medallion architecture)
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
DEFAULT_SCHEMA = "default"

# Volume name for datasets
VOLUME_NAME = "datasets"

# Account group for the M06 GRANT/REVOKE demo (must match TRAINING_ANALYSTS_GROUP in 00_setup)
TRAINING_ANALYSTS_GROUP = "retailhub_analysts"

# Written into the catalog comment in single-user mode; 00_setup reads it (must match 00_setup)
SINGLE_USER_MARKER = "[single-user]"

if not SINGLE_USER_MODE and (not TRAINING_GROUP or TRAINING_GROUP.startswith("<")):
    raise ValueError(
        "Set TRAINING_GROUP to your training group name. "
        "Self-study / Free Edition: set SINGLE_USER_MODE = True instead."
    )

print(f"Mode:             {'SINGLE-USER' if SINGLE_USER_MODE else 'classroom'}")
print(f"Training group:   {'(not used)' if SINGLE_USER_MODE else TRAINING_GROUP}")
print(f"Catalog prefix:   {CATALOG_PREFIX}")
print(f"Storage location: {STORAGE_LOCATION or '(metastore default)'}")

## Step 1: Get Participants from Training Group

Uses the Databricks SDK to read group members, then builds catalog names with `user_slug_for()` --
the same function as in `00_setup`. Trainer logins listed in `TRAINER_ACCOUNTS` (exact match) share `retailhub_trainer`.
Set the same list in `00_setup` in your own Git folder, otherwise your modules look for `retailhub_<login>` and fail with "Catalog not found".

In [0]:
import re
from databricks.sdk import WorkspaceClient

def get_group_members(group_name):
    """
    Get all user members of a Databricks group.
    Returns list of usernames (email addresses).
    The SDK authenticates with the notebook's own credentials, which also works
    on standard (shared) access mode clusters -- unlike the notebook context token.
    """
    w = WorkspaceClient()

    # Find the group by name
    groups = list(w.groups.list(filter=f'displayName eq "{group_name}"'))
    if not groups:
        raise ValueError(f"Group '{group_name}' not found")

    # Get group details with members
    group = w.groups.get(groups[0].id)

    # Resolve each user member to an email
    user_emails = []
    for member in group.members or []:
        if (member.ref or "").startswith("Users/"):
            user = w.users.get(member.value)
            if user.emails:
                user_emails.append(user.emails[0].value)

    return user_emails


# --- BEGIN shared block: keep identical in 00_setup and 00_pre_config ---
# Trainer accounts share one catalog, retailhub_trainer.
# List exact e-mail addresses (or logins before "@"). A "trainer" substring is NOT enough.
TRAINER_ACCOUNTS = []  # e.g. ["trainer@your-company.com"]


def user_slug_for(email):
    """E-mail -> catalog suffix. Example: Jan.Kowalski@example.com -> jan_kowalski."""
    email = email.strip().lower()
    login = email.split("@")[0]
    trainers = {account.strip().lower() for account in TRAINER_ACCOUNTS}
    if email in trainers or login in trainers:
        return "trainer"
    slug = re.sub(r"[^a-z0-9]", "_", login)
    slug = re.sub(r"_+", "_", slug)
    slug = re.sub(r"^[0-9_]+", "", slug)  # a catalog name must not start with a digit
    return slug.strip("_") or "user"
# --- END shared block ---


print("Functions defined: get_group_members(), user_slug_for()")

In [0]:
# =============================================================================
# Get users and build catalog mapping
# =============================================================================
current_email = spark.sql("SELECT current_user()").first()[0]

if SINGLE_USER_MODE:
    training_users = [current_email]
    print("=" * 60)
    print("SINGLE-USER MODE (SINGLE_USER_MODE = True)")
    print(f"Setting up one catalog for {current_email} only.")
    print("For a classroom set SINGLE_USER_MODE = False and TRAINING_GROUP.")
    print("=" * 60)
else:
    try:
        training_users = get_group_members(TRAINING_GROUP)
    except ValueError as e:
        print(f"ERROR: {e}")
        print("  Check the TRAINING_GROUP name (Settings -> Identity and access -> Groups).")
        print("  Self-study / Free Edition: set SINGLE_USER_MODE = True in Step 0.")
        raise
    except Exception as e:
        print(f"ERROR: {type(e).__name__}: {e}")
        print("  Possible issue: you don't have permission to read group members.")
        raise
    if not training_users:
        raise ValueError(f"Group '{TRAINING_GROUP}' has no user members -- add the participants first.")
    print(f"Found {len(training_users)} users in group '{TRAINING_GROUP}':")

print("=" * 60)
user_catalog_map = {}
for email in sorted(training_users):
    catalog_name = f"{CATALOG_PREFIX}_{user_slug_for(email)}"
    user_catalog_map[email] = catalog_name
    print(f"  {email:<40} -> {catalog_name}")

print("=" * 60)
print(f"Total: {len(set(user_catalog_map.values()))} unique catalogs to create")

## Step 2: Create Catalogs, Schemas, and Volumes

For each participant:
- Catalog: `retailhub_{username}` in `STORAGE_LOCATION`, or in the metastore default storage when it is `""`
- Catalog comment: marks single-user mode for `00_setup`
- Schemas: `bronze`, `silver`, `gold`, `default`
- Volume: `datasets` in `default` schema
- Permissions: `ALL PRIVILEGES` granted to the user and the trainer, plus `MANAGE` for the user

The first error stops the notebook with the full message. Every statement is idempotent, so fix the cause and re-run.

> `ALL PRIVILEGES` does **not** include `MANAGE`, and only the owner, a metastore admin or a holder of
> `MANAGE` may grant privileges. The trainer owns every catalog created here, so without `MANAGE`
> participants could not run a single `GRANT` in M06.

In [0]:
def create_user_environment(email, catalog_name):
    """
    Create catalog, schemas, volume and set permissions for a training participant.
    Raises on the first error.
    """
    trainer_email = current_email

    # Catalog: STORAGE_LOCATION if set, metastore default storage only when it is ""
    if STORAGE_LOCATION:
        try:
            spark.sql(f"""
                CREATE CATALOG IF NOT EXISTS {catalog_name}
                MANAGED LOCATION '{STORAGE_LOCATION.rstrip("/")}/{catalog_name}'
            """)
        except Exception as e:
            print(f"  ERROR: cannot create {catalog_name} in STORAGE_LOCATION = {STORAGE_LOCATION}")
            print(f"  {type(e).__name__}: {e}")
            print("  Check the external location and your CREATE MANAGED STORAGE privilege on it,")
            print('  or set STORAGE_LOCATION = "" to use the metastore default storage.')
            raise
    else:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")

    # Mode marker for 00_setup (COMMENT ON also updates catalogs that already exist)
    comment = "RetailHub training catalog" + (f" {SINGLE_USER_MARKER}" if SINGLE_USER_MODE else "")
    spark.sql(f"COMMENT ON CATALOG {catalog_name} IS '{comment}'")

    # Create Medallion schemas + default
    schemas = [DEFAULT_SCHEMA, BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]
    for schema in schemas:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema}")

    # Create managed Volume for datasets
    spark.sql(f"""
        CREATE VOLUME IF NOT EXISTS {catalog_name}.{DEFAULT_SCHEMA}.{VOLUME_NAME}
        COMMENT 'Training datasets for RetailHub project'
    """)

    # Grant permissions
    spark.sql(f"GRANT ALL PRIVILEGES ON CATALOG {catalog_name} TO `{email}`")
    # ALL PRIVILEGES excludes MANAGE -- participants need it to GRANT/REVOKE in M06
    spark.sql(f"GRANT MANAGE ON CATALOG {catalog_name} TO `{email}`")
    if trainer_email != email:
        spark.sql(f"GRANT ALL PRIVILEGES ON CATALOG {catalog_name} TO `{trainer_email}`")

    return {"catalog": catalog_name, "schemas": schemas}

In [0]:
# =============================================================================
# Create environments for all participants (stops on the first error)
# =============================================================================
for email, catalog_name in user_catalog_map.items():
    print(f"Processing: {email}")
    result = create_user_environment(email, catalog_name)
    print(f"  Catalog:     {result['catalog']}")
    print(f"  Schemas:     {', '.join(result['schemas'])}")
    print(f"  Volume:      {VOLUME_NAME}")
    print(f"  Permissions: OK")
    print()

print(f"Result: {len(user_catalog_map)} environments created")

## Step 2b: Check the M06 demo group

Module 06 has every participant grant `retailhub_analysts` read access to their own Gold schema.
That only works if the group is an **account** group -- this cell proves it before the training
instead of letting every participant hit the same error live. Skipped in single-user mode.

In [ ]:
# =============================================================================
# Check that TRAINING_ANALYSTS_GROUP is an account group Unity Catalog can grant to
# =============================================================================
# Workspace-local groups (Workspace Groups API, SQL CREATE GROUP) exist in the
# workspace but not in the metastore's identity store, so GRANT fails with
# PRINCIPAL_DOES_NOT_EXIST. Probe with a grant on one of the catalogs just created
# (the trainer owns them all) and undo it immediately.
if SINGLE_USER_MODE:
    print("INFO: single-user mode -- M06 grants to the built-in `account users` group (resolved in 00_setup).")
else:
    probe_catalog = next(iter(user_catalog_map.values()))
    try:
        spark.sql(f"GRANT USE CATALOG ON CATALOG {probe_catalog} TO `{TRAINING_ANALYSTS_GROUP}`")
        spark.sql(f"REVOKE USE CATALOG ON CATALOG {probe_catalog} FROM `{TRAINING_ANALYSTS_GROUP}`")
        print(f"OK: `{TRAINING_ANALYSTS_GROUP}` is an account group -- M06 demo will work.")
    except Exception as e:
        if "PRINCIPAL_DOES_NOT_EXIST" in str(e):
            print(f"MISSING: `{TRAINING_ANALYSTS_GROUP}` is not visible to Unity Catalog.")
            print("Create it as an account group: Settings -> Identity and access -> Groups")
            print("-> Add Group -> Add new. Do NOT use SQL CREATE GROUP (workspace-local).")
        raise

## Step 3: Copy Dataset Files to Volumes

Copies the entire `dataset/` folder from the Git folder to each user's Volume.
Uses `shutil.copytree` for a complete recursive copy.

**Dataset structure in each Volume:**
```
/Volumes/retailhub_{user}/default/datasets/
  customers/
    customers.csv
    customers_extented.xlsx
    customers_new.csv
  orders/
    orders_batch.json
    stream/
      orders_stream_001.json ... 003.json
  products/
    products.csv
    products.parquet
```

In [0]:
import shutil
import os

def copy_dataset_to_volume(catalog_name):
    """
    Copy dataset files from the Git repo to a user's Volume.
    Skips .DS_Store and other hidden files.
    """
    # Source: dataset/ folder in the repo root.
    # A notebook's working directory is its own folder (notebooks/setup/ in the Git folder),
    # so the repo root is two levels up. Run this notebook from its place in the Git folder.
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
    source_path = os.path.join(repo_root, "dataset")

    # Target: Volume path
    volume_path = f"/Volumes/{catalog_name}/{DEFAULT_SCHEMA}/{VOLUME_NAME}"

    if not os.path.exists(source_path):
        return {"error": f"Source not found: {source_path} -- run 00_pre_config from notebooks/setup/ in the Git folder"}

    try:
        # Copy with ignore for hidden files
        def ignore_hidden(directory, files):
            return [f for f in files if f.startswith('.')]

        shutil.copytree(source_path, volume_path, dirs_exist_ok=True, ignore=ignore_hidden)
        return {"success": True}
    except Exception as e:
        return {"error": str(e)}

In [0]:
# =============================================================================
# Copy dataset to each participant's Volume
# =============================================================================
print("Copying dataset to user Volumes...")
print("=" * 60)

copy_results = {}

for email, catalog_name in user_catalog_map.items():
    print(f"  {catalog_name}... ", end="")
    result = copy_dataset_to_volume(catalog_name)
    copy_results[email] = result

    if "error" in result:
        print(f"FAILED: {result['error']}")
    else:
        print("OK")

successful = sum(1 for r in copy_results.values() if r.get("success"))
print("=" * 60)
print(f"Result: {successful}/{len(copy_results)} volumes populated")

if successful != len(copy_results):
    raise RuntimeError("Dataset copy failed for some catalogs -- see FAILED lines above")

## Step 3b (optional): Solutions for participants who fall behind

The final project lets participants switch to ready solutions after about 25 minutes.
The solutions are **not** in the public repo. To offer them:

1. Before the training, import your local `lab_task_02_silver_solution` and `lab_task_03_gold_solution`
   into a workspace folder of your own (**Workspace → Import**) and set `SOLUTIONS_SOURCE` to that folder.
2. Run the cell below. It copies both notebooks, plus a copy of `00_setup`, to `SOLUTIONS_TARGET_NAME` in your home folder,
   rewrites their `%run` to `./00_setup`, and gives `TRAINING_GROUP` **Can Read** on the folder.
   Participants can open, run and clone the notebooks, or point their job tasks at them, but cannot edit them.

`SOLUTIONS_SOURCE = ""`, or a folder that does not exist, skips this step.

In [ ]:
# =============================================================================
# OPTIONAL: copy lab_task_02/03 solutions to a read-only workspace folder
# =============================================================================
import base64
import json
from databricks.sdk.errors import NotFound
from databricks.sdk.service.iam import AccessControlRequest, PermissionLevel
from databricks.sdk.service.workspace import ExportFormat, ImportFormat

SOLUTION_NOTEBOOKS = ["lab_task_02_silver_solution", "lab_task_03_gold_solution"]


def copy_notebook(source, target, rewrite_run=False):
    """Copy a workspace notebook; optionally point its %run of 00_setup to ./00_setup."""
    content = w.workspace.export(source, format=ExportFormat.JUPYTER).content
    nb_json = json.loads(base64.b64decode(content))
    if rewrite_run:
        for cell in nb_json["cells"]:
            src = cell["source"] if isinstance(cell["source"], list) else [cell["source"]]
            cell["source"] = [re.sub(r"^%run\s+\S*00_setup", "%run ./00_setup", line) for line in src]
    payload = base64.b64encode(json.dumps(nb_json).encode()).decode()
    w.workspace.import_(target, content=payload, format=ImportFormat.JUPYTER, overwrite=True)


w = WorkspaceClient()
source_dir = SOLUTIONS_SOURCE.removeprefix("/Workspace").rstrip("/")
source_ok = False
if source_dir:
    try:
        w.workspace.get_status(source_dir)
        source_ok = True
    except NotFound:
        print(f"SKIPPED: SOLUTIONS_SOURCE not found: {SOLUTIONS_SOURCE}")
else:
    print("SKIPPED: SOLUTIONS_SOURCE is empty.")

if source_ok:
    target_dir = f"/Users/{current_email}/{SOLUTIONS_TARGET_NAME}"
    w.workspace.mkdirs(target_dir)

    # 00_setup from this Git folder (the working directory is notebooks/setup/)
    setup_path = os.getcwd().removeprefix("/Workspace") + "/00_setup"
    copy_notebook(setup_path, f"{target_dir}/00_setup")

    for name in SOLUTION_NOTEBOOKS:
        try:
            copy_notebook(f"{source_dir}/{name}", f"{target_dir}/{name}", rewrite_run=True)
            print(f"  copied: {name}")
        except NotFound:
            print(f"  not found, skipped: {source_dir}/{name}")

    if SINGLE_USER_MODE:
        print(f"Solutions in /Workspace{target_dir} (single-user mode: no group permissions set)")
    else:
        folder_id = w.workspace.get_status(target_dir).object_id
        w.permissions.update(
            request_object_type="directories",
            request_object_id=str(folder_id),
            access_control_list=[
                AccessControlRequest(group_name=TRAINING_GROUP, permission_level=PermissionLevel.CAN_READ)
            ],
        )
        print(f"Solutions in /Workspace{target_dir} -- Can Read for `{TRAINING_GROUP}`")

## Step 4: Verification

Final check that all environments are ready for training.

In [0]:
# =============================================================================
# VERIFICATION -- Check all environments
# =============================================================================
print("=" * 60)
print("TRAINING ENVIRONMENT SUMMARY")
print("=" * 60)
print()

all_ok = True

for email, catalog_name in user_catalog_map.items():
    status = []

    # Check catalog
    try:
        spark.sql(f"USE CATALOG {catalog_name}")
        status.append("catalog: OK")
    except:
        status.append("catalog: MISSING")
        all_ok = False

    # Check schemas
    try:
        schemas = [row[0] for row in spark.sql(f"SHOW SCHEMAS IN {catalog_name}").collect()]
        missing = [s for s in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA] if s not in schemas]
        if missing:
            status.append(f"schemas: MISSING {missing}")
            all_ok = False
        else:
            status.append("schemas: OK")
    except:
        status.append("schemas: ERROR")
        all_ok = False

    # Check Volume
    volume_path = f"/Volumes/{catalog_name}/{DEFAULT_SCHEMA}/{VOLUME_NAME}"
    try:
        files = dbutils.fs.ls(volume_path)
        file_count = len(files)
        if file_count == 0:
            status.append("volume: EMPTY")
            all_ok = False
        else:
            status.append(f"volume: OK ({file_count} items)")
    except:
        status.append("volume: MISSING")
        all_ok = False

    print(f"  {email}")
    print(f"    Catalog: {catalog_name}")
    print(f"    Status:  {' | '.join(status)}")
    print()

print("=" * 60)
if all_ok:
    print("ALL ENVIRONMENTS READY -- Training can begin!")
    print("Every module validates the environment with: %run ../setup/00_setup")
else:
    print("WARNING: Some environments have issues. Check the details above.")
print("=" * 60)

---

## Cleanup (After Training)

Run this section **after** the training to remove all participant catalogs and data.  
**WARNING: This is a destructive operation -- all training data will be permanently deleted!**

In [0]:
# =============================================================================
# CLEANUP -- Remove all training catalogs (after training)
# =============================================================================
# Uncomment the code below to execute cleanup.

# print("Dropping training catalogs...")
# for email, catalog_name in user_catalog_map.items():
#     try:
#         spark.sql(f"DROP CATALOG IF EXISTS {catalog_name} CASCADE")
#         print(f"  Dropped: {catalog_name}")
#     except Exception as e:
#         print(f"  Failed: {catalog_name} -- {e}")
# print("Cleanup complete.")

In [0]:
%skip
# =============================================================================
# ALTERNATIVE CLEANUP -- Find and remove all retailhub_* catalogs
# =============================================================================
# Use this if user_catalog_map is not available (e.g., new session).
# Remove %skip above to list the catalogs; uncomment the loop at the end to drop them.

catalogs_df = spark.sql("SHOW CATALOGS")
retailhub_catalogs = [row.catalog for row in catalogs_df.collect()
                      if row.catalog.startswith("retailhub_")]

print(f"Found {len(retailhub_catalogs)} catalogs to remove:")
for cat in retailhub_catalogs:
    print(f"  - {cat}")

# Uncomment to drop:
# for cat in retailhub_catalogs:
#     spark.sql(f"DROP CATALOG IF EXISTS {cat} CASCADE")
#     print(f"Dropped: {cat}")